# Week 15: Union-Find & Amortized Potential Analysis

**Z5007 Programming and Data Structures (MTech DSAI Core) · IIT Madras Zanzibar**

Potential method verification on binary counters and compressed Disjoint Set Union.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/inyalala/iitmz-teaching-site/blob/main/courses/z5007/week-15/labs/W15_Union_Find_Amortized_Review.ipynb)

Run cells top to bottom (`Runtime > Run all`). Every algorithm is implemented from first principles in Python and verified with assertions.

## 1. Amortized Potential Analysis: Binary Counter & Dynamic Array

With potential function $\Phi(D_i) = \text{number of 1-bits}$, flipping $t_i$ trailing `1`s to `0` and one `0` to `1` incurs actual cost $t_i + 1$ and $\Delta\Phi = 1 - t_i$, so amortized cost is $(t_i + 1) + (1 - t_i) = 2 = O(1)$.

In [1]:
def binary_counter_trace(increments):
    bits = [0] * 32
    total_flips = 0
    for step in range(1, increments + 1):
        i = 0
        while bits[i] == 1:
            bits[i] = 0
            total_flips += 1
            i += 1
        bits[i] = 1
        total_flips += 1
    return total_flips, sum(bits)


for n in (64, 256, 1024, 4096):
    flips, ones = binary_counter_trace(n)
    print(f"n={n:>4} increments -> actual bit flips={flips:>4} ({flips/n:.3f} per increment), final 1-bits={ones}")
    assert flips <= 2 * n


n=  64 increments -> actual bit flips= 127 (1.984 per increment), final 1-bits=1
n= 256 increments -> actual bit flips= 511 (1.996 per increment), final 1-bits=1
n=1024 increments -> actual bit flips=2047 (1.999 per increment), final 1-bits=1
n=4096 increments -> actual bit flips=8191 (2.000 per increment), final 1-bits=1


## 2. Disjoint Set Union: Path Compression + Union by Size

Combining union by size with iterative two-pass path compression keeps average pointer hops per `find` near $O(\alpha(n))$.

In [2]:
class DisjointSetUnion:
    def __init__(self, n):
        self.parent = list(range(n))
        self.size = [1] * n
        self.components = n
        self.hops = 0

    def find(self, x):
        root = x
        while self.parent[root] != root:
            root = self.parent[root]
            self.hops += 1
        while self.parent[x] != root:
            nxt = self.parent[x]
            self.parent[x] = root
            x = nxt
        return root

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return False
        if self.size[ra] < self.size[rb]:
            ra, rb = rb, ra
        self.parent[rb] = ra
        self.size[ra] += self.size[rb]
        self.components -= 1
        return True


dsu = DisjointSetUnion(500)
for i in range(499):
    dsu.union(i, i + 1)
for i in range(500):
    assert dsu.find(i) == dsu.find(0)
print(f"500-node chain compressed: components={dsu.components}, total hops={dsu.hops}")


500-node chain compressed: components=1, total hops=997
